# [10.1] Capstone Research Sprint - Solutions

This notebook mirrors the learner exercises with solved helper implementations, then audits the committed CUDA mini activation-oracle report. The production release hook lives in `solutions.run_gpu_test` and reruns the capstone script on CUDA.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

chapter = "chapter10_capstone_research_sprint"
section = "part1_capstone_research_sprint"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_capstone_research_sprint.tests as tests
import part1_capstone_research_sprint.utils as utils
from chapter10_capstone_research_sprint.exercises.part1_capstone_research_sprint import solutions


In [ ]:
CapstonePlan = solutions.CapstonePlan
BaselineSuiteReport = solutions.BaselineSuiteReport
CausalValidationSuiteReport = solutions.CausalValidationSuiteReport
ReproducibilityReport = solutions.ReproducibilityReport
CapstoneReadinessReport = solutions.CapstoneReadinessReport


## Planning Contract

A capstone plan should be normalized before any experiment runs. Blank copied fields are dropped, repeated fields become tuples, and the written claim stays explicit.


In [ ]:
def build_capstone_plan(
    *,
    research_question: str,
    benchmark: str,
    baselines: list[str],
    mechanistic_claim: str,
    causal_validations: list[str],
    reproducible_scripts: list[str],
    writeup_path: str,
) -> CapstonePlan:
    return CapstonePlan(
        research_question=research_question.strip(),
        benchmark=benchmark.strip(),
        baselines=tuple(baseline.strip() for baseline in baselines if baseline.strip()),
        mechanistic_claim=mechanistic_claim.strip(),
        causal_validations=tuple(
            validation.strip()
            for validation in causal_validations
            if validation.strip()
        ),
        reproducible_scripts=tuple(
            script.strip()
            for script in reproducible_scripts
            if script.strip()
        ),
        writeup_path=writeup_path.strip(),
    )


tests.test_build_capstone_plan_normalizes_blank_fields(build_capstone_plan)


## Baselines And Validations

The readiness scaffold is intentionally strict about missing baselines and causal checks. It should name the missing pieces rather than return a vague failure.


In [ ]:
def baseline_suite_report(
    present_baselines: list[str],
    *,
    required_baselines: tuple[str, ...] = ("probe", "text_only", "random_control"),
) -> BaselineSuiteReport:
    present = tuple(baseline.strip() for baseline in present_baselines if baseline.strip())
    present_set = set(present)
    missing = tuple(
        baseline
        for baseline in required_baselines
        if baseline not in present_set
    )
    return BaselineSuiteReport(
        required_baselines=required_baselines,
        present_baselines=present,
        missing_baselines=missing,
        complete=len(missing) == 0,
    )


def causal_validation_suite_report(
    validations: list[str],
) -> CausalValidationSuiteReport:
    normalized = tuple(validation.strip().lower() for validation in validations)
    validation_set = set(normalized)
    has_ablation = "ablation" in validation_set
    has_patching = "patching" in validation_set or "counterfactual_patching" in validation_set
    has_random_control = "random_control" in validation_set
    has_ood = "ood" in validation_set or "heldout_templates" in validation_set
    complete = has_ablation and has_patching and has_random_control and has_ood
    return CausalValidationSuiteReport(
        validations=normalized,
        has_ablation=has_ablation,
        has_patching=has_patching,
        has_random_control=has_random_control,
        has_ood=has_ood,
        complete=complete,
    )


tests.test_baseline_suite_report_identifies_missing_required_baseline(baseline_suite_report)
tests.test_baseline_smoke_test_has_required_controls(solutions.baseline_smoke_test)
tests.test_causal_validation_suite_report_accepts_equivalent_names(causal_validation_suite_report)


## Reproducibility And Readiness

A script path is only useful if it exists relative to the section root, and an artifact path is only useful if the run produced it.


In [ ]:
def reproducibility_report(
    *,
    script_paths: list[str],
    seeds: list[int],
    artifact_paths: list[str],
    root: str | Path | None = None,
) -> ReproducibilityReport:
    scripts = tuple(path.strip() for path in script_paths if path.strip())
    artifacts = tuple(path.strip() for path in artifact_paths if path.strip())
    seed_tuple = tuple(int(seed) for seed in seeds)
    root_path = Path.cwd() if root is None else Path(root)
    scripts_exist = all((root_path / script).is_file() for script in scripts)
    artifacts_exist = all((root_path / artifact).is_file() for artifact in artifacts)
    return ReproducibilityReport(
        script_paths=scripts,
        seeds=seed_tuple,
        artifact_paths=artifacts,
        reproducible=bool(scripts and seed_tuple and artifacts and scripts_exist and artifacts_exist),
    )


def capstone_readiness_report(
    plan: CapstonePlan,
    baselines: BaselineSuiteReport,
    validations: CausalValidationSuiteReport,
    reproducibility: ReproducibilityReport,
) -> CapstoneReadinessReport:
    has_question = bool(plan.research_question)
    has_benchmark = bool(plan.benchmark)
    has_claim = bool(plan.mechanistic_claim)
    has_writeup = bool(plan.writeup_path)
    ready = (
        has_question
        and has_benchmark
        and has_claim
        and baselines.complete
        and validations.complete
        and reproducibility.reproducible
        and has_writeup
    )
    return CapstoneReadinessReport(
        has_research_question=has_question,
        has_benchmark=has_benchmark,
        has_mechanistic_claim=has_claim,
        baseline_suite_complete=baselines.complete,
        causal_validation_complete=validations.complete,
        reproducibility_complete=reproducibility.reproducible,
        has_writeup_path=has_writeup,
        ready=ready,
    )


tests.test_reproducibility_report_requires_scripts_seeds_and_artifacts(reproducibility_report)
tests.test_capstone_readiness_report_requires_every_gate(
    build_capstone_plan,
    baseline_suite_report,
    causal_validation_suite_report,
    reproducibility_report,
    capstone_readiness_report,
)


## Notebook Contract

The local smoke contract remains planning-focused. The live experiment evidence is checked in the CUDA report cell below.


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["plan"]["research_question"] == "Do mini Activation Oracles beat probes?"
assert contract["plan"]["benchmark"] == "held-out activation questions"
assert contract["baselines"]["complete"]
assert contract["validations"]["complete"]
assert contract["reproducibility"]["reproducible"]
assert contract["readiness"]["ready"]
tests.test_notebook_contract(solutions.run_smoke_test)


## CUDA Mini-Capstone Report

This cell audits the committed report produced by `scripts/run_capstone.py`. It checks the live training metrics, baseline deltas, causal controls, per-seed file validation, and VRAM budget.


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert report["gt_tier"] == "GT-4"
assert report["notebook_id"] == "10_1_capstone_research_sprint"
assert gpu["cuda_available"]
assert gpu["live_training_executed"]
assert gpu["seed_count"] == 3
assert gpu["oracle_accuracy_mean"] >= 0.9
assert gpu["oracle_beats_text_only"]
assert gpu["oracle_beats_linear_probe_bank"]
assert gpu["compositional_oracle_beats_linear_probe"]
assert gpu["heldout_template_accuracy_mean"] >= 0.9
assert gpu["ablation_drop_mean"] > 0.2
assert gpu["counterfactual_patch_target_accuracy_mean"] >= 0.9
assert gpu["random_patch_change_rate_mean"] <= 0.15
assert gpu["random_activation_control_passed"]
assert gpu["label_shuffle_control_passed"]
assert gpu["metrics_by_seed_file_valid"]
assert gpu["preflight_passed"]
assert gpu["peak_vram_gb"] <= 1.0
utils.print_report("10.1 CUDA mini-capstone report", {
    "device": gpu["device"],
    "seed_count": gpu["seed_count"],
    "oracle_accuracy_mean": gpu["oracle_accuracy_mean"],
    "linear_probe_compositional_accuracy_mean": gpu["linear_probe_compositional_accuracy_mean"],
    "ablation_drop_mean": gpu["ablation_drop_mean"],
    "peak_vram_gb": gpu["peak_vram_gb"],
})
